# Custom_Reviewer_Example

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import os

## Set up Data

In [3]:
# df = pd.read_csv('load_example_data.tsv', sep='\t')

## Run Reviewer

In [4]:
from SNVReviewers.Reviewers.SNVReviewer import SNVReviewer

In [5]:
import os
import sys
from SNVReviewers.AppComponents.utils import generate_dig_report_dataframe_combined, generate_coding_region_dig_dataframe

# put into utils file later!!

# MAKE SURE TO PUT THE NEW DIG/MUTSIG/DNDSCV Report Data into the folder and access it locally

data_folder_path = "./data"
dig_data_filenames = []
mutsig_data_filenames = []
dndscv_data_filenames = []
snv_data_foldernames = []

for foldername in os.listdir(data_folder_path):
    folder_path = data_folder_path + "/" + foldername
    snv_data_foldernames.append(folder_path)
    print(f"folder_path: {folder_path}")

for snv_data_folder in snv_data_foldernames:

    for filename in os.listdir(snv_data_folder):

        # only want to read txt files
        if ".txt" in filename:
            new_file_name = snv_data_folder + "/" + filename
            
            if "dig" in filename:
                
                dig_data_filenames.append(new_file_name)
                
        
            elif "dnds" in filename:
                # new_file_name = snv_data_folder + "/" + filename
                dndscv_data_filenames.append(new_file_name)

            elif "sig" in filename:
                # new_file_name = snv_data_folder + "/" + filename
                mutsig_data_filenames.append(new_file_name)

# MIGHT NOT NEED TO DO THIS BECAUSE THE DIG_TOOLS READS IN THE TEXT FILE 
# using generate_dig_report_coding file using generate_dig_report function

dig_coding_file_path = ""
dig_combined_file_path = ""
# read in the data to panda dataframe
for txt_file in dig_data_filenames:
    # combined dig report results does not have the 'OBS_NONSYN' column
    # gets the coding dig report results
    if 'coding' in txt_file:
        dig_coding_file_path = txt_file
        print(f"dig_data_file_path: {dig_coding_file_path}")

    elif 'combined' in txt_file:
        dig_combined_file_path = txt_file
        print(f"combined data file: {dig_combined_file_path}")

# for txt_file in dndscv_data_filenames:
#     dnds_data = #open(txt_file, "r").read()

# for txt_file in mutsig_data_filenames:
#     mutsig_data = #open(txt_file, "r").read()

folder_path: ./data/dndscv_results_new
folder_path: ./data/combined_results
folder_path: ./data/mutsig_reports_folder
folder_path: ./data/mutsig2_results_new
folder_path: ./data/dig_results_new
folder_path: ./data/dNdScv_reports_folder
folder_path: ./data/Dig_reports_folder
folder_path: ./data/reviewer_data
combined data file: ./data/dig_results_new/ov100-hg38.combined.dig.results.txt
dig_data_file_path: ./data/dig_results_new/ov100-hg38.coding.dig.results.txt
dig_data_file_path: ./data/Dig_reports_folder/dlbcl-july-2024.coding.dig.results.txt
combined data file: ./data/Dig_reports_folder/dlbcl-july-2024.combined.dig.results.txt


In [6]:
# cgc_list_path = "gs://getzlab-workflows-reference_files-oa/hg19/dig/cancer_gene_census_2024_06_20.tsv"
# pancan_list_path = "gs://getzlab-workflows-reference_files-oa/hg19/dig/pancanatlas_genes.tsv"

# cgc_list = pd.read_csv(cgc_list_path, sep='\t').to_numpy().flatten()
# pancan_list = pd.read_csv(pancan_list_path, sep='\t').to_numpy().flatten()

In [7]:
# dig_combined_df = generate_dig_report_dataframe_combined(dig_combined_file_path)
# dig_combined_df = generate_dig_report_dataframe_combined(dig_coding_file_path)
dig_coding_df = generate_coding_region_dig_dataframe(dig_coding_file_path)
# dig_coding_gene_chrom = dig_coding_df[['GENE', 'CHROM']]

dig_combined_df = pd.read_csv(dig_combined_file_path, sep='\t')
# dig_coding_df = pd.read_csv(dig_coding_file_path, sep='\t')

# if duplicate column names then
dig_df = dig_combined_df.merge(dig_coding_df, on='GENE', how='outer', suffixes=('_combined', '_coding'))
# dig_df = dig_combined_df.merge(dig_coding_df, on='GENE', how='inner')
# dig_df = dig_combined_df.merge(dig_coding_df, on='GENE', how='left')
# dig_df = dig_combined_df.merge(dig_coding_df, on='GENE', how='right')

if 'PANCAN_combined' in list(dig_df.columns):
    print("Found column in dig_df")
    dig_df = dig_df.drop('PANCAN_combined', axis=1)
    dig_df = dig_df.drop('CGC_combined', axis=1)
    dig_df = dig_df.rename(columns={'CGC_coding': 'CGC'})
    dig_df = dig_df.rename(columns={'PANCAN_coding': 'PANCAN'})
    
# print(list(dig_df.columns))
mutsig_df = pd.DataFrame()
dnd_df = pd.DataFrame()
cohort_name = "dlbcl"

df = pd.DataFrame({"cohort": [cohort_name], "snv_data": [ [dig_df, dnd_df, mutsig_df] ]}, index=[cohort_name])#.reset_index("cohort")
# make a new dataframe with one row, corresponding to the cohort -> dlbcl and the data corresponding to that cohort (all 3 reports)
    # make one column with index -> cohort
    # make second column with data -> contains a list of dataframes [dig_df, mutsig_df, dndscv_df] # make global inddex to be able to access those df

Found column in dig_df


In [8]:
print("Number of rows in dig_coding_df: ", len(dig_coding_df))
print("Number of rows in dig_combined_df: ", len(dig_combined_df))
print("Number of rows in final combined dig_df:", len(dig_df))
print("Number of columns in dig combined df: ", len(list(dig_combined_df.columns)))
print("Number of columns in dig combined df: ", len(list(dig_coding_df.columns)))
print("Number of columns in merged dataframe: ", len(list(dig_df.columns)))
print("CGC in dig_df: ", 'CGC' in list(dig_df.columns))
print("PANCAN in dig_df: ", 'PANCAN' in list(dig_df.columns))
print("CGC in dig_combined_df: ", 'CGC' in list(dig_combined_df.columns))
print("CGC in dig_coding_df: ", 'PANCAN' in list(dig_coding_df.columns))
print('PVAL_NONSYN_BURDEN_unif in dig_df: ', 'PVAL_NONSYN_BURDEN_unif' in list(dig_df.columns)) 


Number of rows in dig_coding_df:  19210
Number of rows in dig_combined_df:  19210
Number of rows in final combined dig_df: 19210
Number of columns in dig combined df:  103
Number of columns in dig combined df:  117
Number of columns in merged dataframe:  217
CGC in dig_df:  True
PANCAN in dig_df:  True
CGC in dig_combined_df:  True
CGC in dig_coding_df:  True
PVAL_NONSYN_BURDEN_unif in dig_df:  True


In [9]:
# print("Columns in dig combined df: ", len(list(dig_combined_df.columns)))
# print("Columns in dig combined df: ", len(list(dig_coding_df.columns)))
# print("Columns in merged dataframe: ", len(list(dig_df.columns)))

In [10]:
df.head()
print("Number of rows in dataframe: ", len(df))
data_path = './data/reviewer_data/snv_reviewer.pkl'

Number of rows in dataframe:  1


In [11]:
snv_reviewer = SNVReviewer()
snv_reviewer.set_review_data(
    data_path = data_path, 
    description='SNV Reviewer', 
    df=df,
    index=df.index,
    # index="cohort"
)
snv_reviewer.set_review_app()
snv_reviewer.set_default_review_data_annotations_configuration()
# snv_reviewer.set_default_autofill()

In [12]:
snv_reviewer.run(port=8099, mode="tab")
# use the components_name_order attribute in the AnnoMate run()

Setting auto_export_path to ./data/reviewer_data/snv_reviewer.pkl/data.auto_export
Using ./data/reviewer_data/snv_reviewer.pkl/data.auto_export for auto exporting.
Dash app running on http://0.0.0.0:8099/


/Users/odias/SNVReviewer/snv_venv/lib/python3.9/site-packages/AnnoMate/ReviewDataApp.py:738: UserWarning:

You are in test mode. Your data will not be saved.



<IPython.core.display.Javascript object>

In [13]:
# import pandas as pd
# csc_df = pd.read_csv("gs://getzlab-workflows-reference_files-oa/hg19/dig/cancer_gene_census_2024_06_20.tsv", sep="\t")
# pancan_df = pd.read_csv("gs://getzlab-workflows-reference_files-oa/hg19/dig/pancanatlas_genes.tsv", sep="\t")